In [ ]:
pip install ipynb

In [ ]:
import tkinter as tk
from tkinter import messagebox, ttk
import csv
import os
from ipynb.fs.full.data_manager import save_application, get_applications, get_job_applicants

def get_job_by_id(job_id):
    if os.path.exists("jobs.csv"):
        with open("jobs.csv", "r") as file:
            reader = csv.DictReader(file)
            for row in reader:
                if row["id"] == str(job_id):
                    return row
    return None

class WorkLinkApp:

    def __init__(self, root):
        self.root = root
        self.root.title("WorkLink - Application Management Portal")
        self.root.geometry("600x500")

        self.current_user = {
            "id": "2",  # Example ID
            "name": "Jane Doe",
            "role": "Job Seeker", 
        }

        header = tk.Label(
            root,
            text=f"WorkLink Portal - Logged in as: {self.current_user['name']} ({self.current_user['role']})",
            font=("Arial", 12, "bold"),
            bg="#2c3e50",
            fg="white",
            pady=10,
        )
        header.pack(fill=tk.X)

        self.main_frame = tk.Frame(root, padx=20, pady=20)
        self.main_frame.pack(fill=tk.BOTH, expand=True)

        if self.current_user["role"] == "Job Seeker":
            self.build_job_seeker_dashboard()
        else:
            self.build_employer_dashboard()

    def build_job_seeker_dashboard(self):
        """Creates the primary hub for Job seekers to apply and view history."""
        # Clean workspace
        for widget in self.main_frame.winfo_children():
            widget.destroy()

        self.selected_job_id = "101"

        title_lbl = tk.Label(
            self.main_frame,
            text=f"Selected Position ID: {self.selected_job_id}",
            font=("Arial", 11, "bold"),
        )
        title_lbl.pack(anchor="w", pady=(0, 5))

        apply_btn = tk.Button(
            self.main_frame,
            text="Apply to Job",
            bg="#27ae60",
            fg="white",
            font=("Arial", 10, "bold"),
            command=self.handle_apply,
        )
        apply_btn.pack(fill=tk.X, pady=5)

        ttk.Separator(self.main_frame, orient="horizontal").pack(
            fill=tk.X, pady=20
        )

        history_lbl = tk.Label(
            self.main_frame,
            text="My Submitted Applications:",
            font=("Arial", 11, "bold"),
        )
        history_lbl.pack(anchor="w")

        self.app_tree = ttk.Treeview(
            self.main_frame, columns=("Job ID", "Title"), show="headings"
        )
        self.app_tree.heading("Job ID", text="Job ID")
        self.app_tree.heading("Title", text="Job Title")
        self.app_tree.column("Job ID", width=80, anchor="center")
        self.app_tree.pack(fill=tk.BOTH, expand=True, pady=10)

        self.populate_my_applications()

    def handle_apply(self):
        """Action handler that executes backend application storage."""
        employee_id = self.current_user["id"]
        job_id = self.selected_job_id

        success = save_application(job_id, employee_id)

        if success:
            messagebox.showinfo(
                "Success", "Your application has been submitted successfully!"
            )
            self.populate_my_applications() 
        else:
            messagebox.showwarning(
                "Duplicate Application",
                "You have already applied for this position.",
            )

    def populate_my_applications(self):
        """Clears old rows and fetches fresh records matching current candidate."""
        for item in self.app_tree.get_children():
            self.app_tree.delete(item)

        my_apps = get_applications(self.current_user["id"])

        for app in my_apps:
            job_details = get_job_by_id(app["job_id"])
            job_title = (
                job_details["title"] if job_details else "Sample Job Position"
            )

            self.app_tree.insert(
                "", "end", values=(app["job_id"], job_title)
            )
    def build_employer_dashboard(self):
        """Builds interface for Employer to view submissions based on job IDs."""
        for widget in self.main_frame.winfo_children():
            widget.destroy()

        lbl = tk.Label(
            self.main_frame,
            text="Enter a Job ID to review active applicants:",
            font=("Arial", 11),
        )
        lbl.pack(anchor="w", pady=5)

        self.job_id_entry = tk.Entry(self.main_frame, font=("Arial", 11))
        self.job_id_entry.insert(0, "101")  # Pre-fill for testing purposes
        self.job_id_entry.pack(fill=tk.X, pady=5)

        view_applicants_btn = tk.Button(
            self.main_frame,
            text="Fetch Applicants List",
            bg="#2980b9",
            fg="white",
            command=self.show_applicants_window,
        )
        view_applicants_btn.pack(fill=tk.X, pady=10)

    def show_applicants_window(self):
        """Creates a standalone secondary viewing dashboard displaying candidate metrics."""
        target_job_id = self.job_id_entry.get().strip()
        if not target_job_id:
            messagebox.showerror("Error", "Please specify a valid Job ID.")
            return

        profiles = get_job_applicants(target_job_id)

        viewer_win = tk.Toplevel(self.root)
        viewer_win.title(f"Applicants List - Job Reference #{target_job_id}")
        viewer_win.geometry("550x350")

        header_lbl = tk.Label(
            viewer_win,
            text=f"Total Candidates: {len(profiles)}",
            font=("Arial", 11, "bold"),
            pady=10,
        )
        header_lbl.pack()

        details_tree = ttk.Treeview(
            viewer_win,
            columns=("ID", "Name", "Email", "Skills"),
            show="headings",
        )
        details_tree.heading("ID", text="User ID")
        details_tree.heading("Name", text="Full Name")
        details_tree.heading("Email", text="Email Contact")
        details_tree.heading("Skills", text="Skills Breakdown")

        details_tree.column("ID", width=60, anchor="center")
        details_tree.column("Name", width=120)
        details_tree.column("Email", width=140)
        details_tree.column("Skills", width=200)

        details_tree.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)

        for prof in profiles:
            details_tree.insert(
                "",
                "end",
                values=(
                    prof["id"],
                    prof["name"],
                    prof["email"],
                    prof["skills"],
                ),
            )


# Runtime initialization
if __name__ == "__main__":
    # Create empty mock data files dynamically if testing as a standalone app blueprint
    if not os.path.exists("users.csv"):
        with open("users.csv", "w", newline="") as f:
            w = csv.writer(f)
            w.writerow(["id", "name", "email", "password", "role", "skills"])
            w.writerow(
                [
                    "2",
                    "Jane Doe",
                    "jane@test.com",
                    "pass",
                    "Job Seeker",
                    "Python, Tkinter, SQL",
                ]
            )

    root = tk.Tk()
    app = WorkLinkApp(root)
    root.mainloop()